In [ ]:
# PEPTIDE SEQUENCING ANALYSIS PIPELINE
# Complete Analysis with All Visualizations

# Import required packages
from pathlib import Path
import csv
import re
import gc
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d
from matplotlib.ticker import AutoMinorLocator  
from pyteomics import mass
from collections import Counter
from scipy.stats import gaussian_kde
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
import logomaker
import matplotlib as mpl
import matplotlib_venn
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import FixedLocator

# ================================================================
# MATPLOTLIB CONFIGURATION
# ================================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi": 300
})

aa_colors = {
    'W': "#560030", 'F': "#B50D6C", 'Y': "#CE4693",
    'A': '#E66101', 'V': "#A34A0A", 'L': "#7D3908", 'I': "#5B2804",
    'K': "#1978AF", 'R': "#032740", 'H': "#76BDE7",
    'D': '#7C131C', 'E': "#DA4760",
    'S': '#1A9850', 'T': "#26C46B", 'N': "#085719", 'Q': "#A7F0C7",
    'G': '#CCCCCC', 'P': '#CCCCCC', 'C': '#CCCCCC', 'M': '#CCCCCC'
}

# ================================================================
# CONFIGURATION SECTION
# ================================================================

base_path = Path(r'./data') # Data path relative to script location
input_file = base_path / 'denovoAllCandidates.csv'
output_file = base_path / 'filtered-peptides-allowedletters.csv'
denovo_file = base_path / 'Sample 1.denovo.csv'
save_dir = base_path
output_stats_csv = base_path / 'output_statistics.csv'
output_tps_csv = base_path / 'true_positives.csv'
output_exp_peptides_csv = base_path / 'experimental_peptides.csv'
theoretical_csv = base_path / 'Lib2_theoretical.csv'

# Filter settings, examplewise given for library No2: XXK, X = All aa except I and C
length_criteria = 3  # length of the peptide sequence
ptm_positions = [2]  # position of PTMs, indexed 0
specific_letters = ['K']   # fixed amino acid (if applicable)
specific_positions = [2]  # position of fixed amino acid, indexed 0
alc_threshold = 0   # ALC threshold

# Allowed letters and positions
allowed_letters = {'A', 'R', 'N', 'D', 'E', 'Q', 'G', 'H', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V'}  # amino acids used in variable positions in library
allowed_positions = {0, 1}  # variable amino acid positions in library, indexed 0

# Amino acids to track
amino_acids_to_track = ['A', 'R', 'N', 'D', 'E', 'Q', 'G', 'H', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V'] # amino acids used in variable and FIXED positions in library
amino_acids_at_VARIABLE_positions = ['A', 'R', 'N', 'D', 'E', 'Q', 'G', 'H', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V'] # amino acids used in variable positions in library
variable_positions = [1, 2] # variable amino acid positions in library, indexed 1

# Plot settings
dpi_setting = 300
plot_size_setting = (5, 8)
font_size_setting = 8

# ================================================================
# UTILITY FUNCTIONS
# ================================================================

def ensure_directory(directory):
    """Create directory if it doesn't exist"""
    if not os.path.exists(directory):
        os.makedirs(directory)

def process_peptide(peptide):
    """Extract amino acids and PTMs from peptide string"""
    result = []
    pattern = r'([A-Z])((?:\(\+?[\d\.-]+\))+)?'   
    matches = re.findall(pattern, peptide)
    index = 0
    
    for match in matches:
        amino_acid = match[0]
        ptms = match[1]
        ptm_masses = re.findall(r'[\d\.-]+', ptms) if ptms else []
        ptm_list = [{'PTM': f"({ptm_mass})", 'PTM_mass': float(ptm_mass)} for ptm_mass in ptm_masses]
        result.append({'letter': amino_acid, 'PTMs': ptm_list, 'position': index})
        index += 1
    
    return result

def check_specific_letters_positions(peptide, allowed_letters, allowed_positions):
    """Ensures that only specific letters appear in the defined positions"""
    for pos in allowed_positions:
        if pos >= len(peptide):  
            return False
        if peptide[pos]['letter'] not in allowed_letters:  
            return False
    return True

def clean_sequence(sequence):
    """Remove any characters in parentheses"""
    return re.sub(r'\(.*?\)', '', sequence)

def clean_peptides(data):
    """Extract amino acids from peptide strings"""
    if isinstance(data, pd.DataFrame):
        cleaned_peptides = data['Peptide'].str.replace(r'[^A-Za-z]', '', regex=True)
        return cleaned_peptides
    elif isinstance(data, list):
        cleaned_peptides = [pep['Peptide'].replace(r'[^A-Za-z]', '') for pep in data if 'Peptide' in pep]
        return cleaned_peptides
    else:
        raise ValueError("Unsupported data type.")

def load_data(output_file):
    """Load data from CSV file"""
    data = pd.read_csv(output_file)
    return data.to_dict(orient='records')

def create_custom_color_scheme():
    """Create custom color scheme dictionary"""
    return aa_colors

# ================================================================
# PEPTIDE FILTERING
# ================================================================

def filter_peptides(input_file, output_file, length_criteria, ptm_positions, specific_letters=None, 
                   specific_positions=None, allowed_letters=None, allowed_positions=None):
    """Filter peptides based on multiple criteria"""
    seen_sequences = set()  
    
    with open(input_file, mode='r') as infile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames
        
        with open(output_file, mode='w', newline='') as outfile:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()
            
            for row in reader:
                peptide = row['Peptide']  
                processed_peptide = process_peptide(peptide)
                
                if len(processed_peptide) != length_criteria:
                    continue
                
                if len(ptm_positions) > 0:
                    ptm_positions_found = [aa['position'] for aa in processed_peptide if aa['PTMs']]
                    if not all(pos in ptm_positions_found for pos in ptm_positions) or len(ptm_positions_found) != len(ptm_positions):
                        continue  
                
                if specific_letters and specific_positions:
                    all_letters_match = all(
                        processed_peptide[pos]['letter'] == letter for letter, pos in zip(specific_letters, specific_positions)
                    )
                    if not all_letters_match:
                        continue
                
                if allowed_letters and allowed_positions:
                    if not check_specific_letters_positions(processed_peptide, allowed_letters, allowed_positions):
                        continue
                
                if alc_threshold is not None:
                    alc_value = float(row['ALC (%)'])  
                    if alc_value < alc_threshold: 
                        continue  

                peptide_sequence = ''.join([aa['letter'] for aa in processed_peptide])
                if peptide_sequence in seen_sequences:
                    continue  
                
                seen_sequences.add(peptide_sequence)
                writer.writerow(row)
                
                del processed_peptide
                gc.collect()

print("\n" + "="*60)
print("FILTERING PEPTIDES...")
print("="*60)
filter_peptides(input_file, output_file, length_criteria, ptm_positions, specific_letters, 
               specific_positions, allowed_letters, allowed_positions)
print("✓ Peptide filtering complete\n")

# ================================================================
# MASS CALCULATION
# ================================================================

beta_mass_additions = {}
default_addition = 0

beta_amino_acid_masses = {
    aa: mass.std_aa_mass[aa] + beta_mass_additions.get(aa, default_addition)
    for aa in mass.std_aa_mass
}

def calculate_mass(peptide):
    """Calculate theoretical peptide mass with amidation"""
    total_mass = sum(beta_amino_acid_masses.get(residue, 0) for residue in peptide)
    return (
        total_mass 
        + mass.Composition(formula='OH').mass()  
        + mass.Composition(formula='H').mass() # replace 'H' with 'C8H16O' for Octanoyl N-term acylation 
        - mass.Composition(formula='OH').mass()  
        + mass.Composition(formula='NH2').mass()
    )

def load_theoretical_masses(output_file, output_csv):
    """Load and calculate theoretical masses"""
    data = pd.read_csv(output_file)
    data['Mass'] = data['Peptide'].apply(calculate_mass)
    data.to_csv(output_csv, index=False)
    print(f"✓ Theoretical masses saved to {output_csv}")
    return data['Mass'].tolist()

# ================================================================
# MASS DISTRIBUTION PLOTS
# ================================================================

def MassDistribution_KDE_Improved(DataList, theoretical_masses, save_dir, FigureName, raw=False, dpi=300):
    """KDE plot with unbinned data visualization"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    MassVect = np.array([Pep['Mass'] for Pep in DataList])
    theoretical_masses = np.array(theoretical_masses)

    Q1_exp, Q3_exp = np.percentile(MassVect, [25, 75])
    IQR_exp = Q3_exp - Q1_exp
    exp_filtered = MassVect[(MassVect >= Q1_exp - 1.5*IQR_exp) & (MassVect <= Q3_exp + 1.5*IQR_exp)]

    fig, ax = plt.subplots(figsize=(10, 6), dpi=dpi)

    kde_exp = gaussian_kde(exp_filtered)
    kde_theo = gaussian_kde(theoretical_masses)

    x_grid = np.linspace(
        min(exp_filtered.min(), theoretical_masses.min()) * 0.98,
        max(exp_filtered.max(), theoretical_masses.max()) * 1.02,
        1000
    )

    y_exp = kde_exp(x_grid)
    y_theo = kde_theo(x_grid)

    jitter_height_exp = np.max(y_exp) * 0.02
    jitter_height_theo = np.max(y_theo) * 0.02
    
    ax.plot(exp_filtered, np.zeros_like(exp_filtered), '|',
            color='#8C564B', markersize=12, alpha=0.5, linewidth=1.5, label='Experimental (observed)')
    ax.plot(theoretical_masses, -jitter_height_theo * np.ones_like(theoretical_masses), '|',
            color='#4C72B0', markersize=12, alpha=0.5, linewidth=1.5, label='Theoretical (expected)')

    ax.plot(x_grid, y_exp, color='#8C564B', linewidth=2.5)
    ax.plot(x_grid, y_theo, color='#4C72B0', linewidth=2.5)

    y_min = np.minimum(y_exp, y_theo)
    ax.fill_between(x_grid, y_min, alpha=0.3, color='#FF7F0E', label='Overlap')

    overlap_area = np.trapz(y_min, x_grid)
    total_area_exp = np.trapz(y_exp, x_grid)
    overlap_percentage = (overlap_area / total_area_exp) * 100

    ax.set_xlabel('Mass (Da)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Density', fontsize=14, fontweight='bold')
    ax.set_title('Mass Distribution (Unbinned)', fontsize=16, fontweight='bold')

    stats_text = (f"N(exp) = {len(exp_filtered)}\n"
                  f"N(theo) = {len(theoretical_masses)}\n"
                  f"Overlap = {overlap_percentage:.1f}%")
    ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
            fontsize=11, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ KDE overlap: {overlap_percentage:.2f}%")
    
    return overlap_percentage


def MassDistribution_Violin(DataList, theoretical_masses, save_dir, FigureName, raw=False, dpi=300):
    """Violin plot showing distribution shape"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    MassVect = np.array([Pep['Mass'] for Pep in DataList])
    theoretical_masses = np.array(theoretical_masses)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=dpi)

    parts = ax.violinplot([MassVect, theoretical_masses], positions=[1, 2],
                           showmeans=True, showmedians=True)
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['#8C564B', '#4C72B0'][i])
        pc.set_alpha(0.7)

    y_exp = np.random.normal(1, 0.04, size=len(MassVect))
    y_theo = np.random.normal(2, 0.04, size=len(theoretical_masses))
    
    ax.scatter(y_exp, MassVect, alpha=0.3, s=30, color='#8C564B')
    ax.scatter(y_theo, theoretical_masses, alpha=0.3, s=30, color='#4C72B0')

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Experimental', 'Theoretical'], fontsize=12)
    ax.set_ylabel('Mass (Da)', fontsize=14, fontweight='bold')
    ax.set_title('Mass Distribution (Violin Plot)', fontsize=16, fontweight='bold')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Violin plot saved")


def MassDistribution_Cumulative(DataList, theoretical_masses, save_dir, FigureName, raw=False, dpi=300):
    """Cumulative distribution function"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    MassVect = np.array(sorted([Pep['Mass'] for Pep in DataList]))
    theoretical_masses = np.array(sorted(theoretical_masses))

    fig, ax = plt.subplots(figsize=(8, 6), dpi=dpi)

    ax.plot(MassVect, np.arange(1, len(MassVect)+1)/len(MassVect), 
            marker='.', linestyle='-', color='#8C564B', alpha=0.7, 
            markersize=4, label='Experimental', linewidth=2)
    
    ax.plot(theoretical_masses, np.arange(1, len(theoretical_masses)+1)/len(theoretical_masses),
            marker='.', linestyle='-', color='#4C72B0', alpha=0.7,
            markersize=4, label='Theoretical', linewidth=2)

    ax.set_xlabel('Mass (Da)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Probability', fontsize=14, fontweight='bold')
    ax.set_title('Cumulative Mass Distribution', fontsize=16, fontweight='bold')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Cumulative distribution saved")


def MassError_Distribution(DataList, save_dir, FigureName, dpi=300):
    """Distribution of mass errors"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    ppm_errors = np.array([Pep['ppm'] for Pep in DataList if 'ppm' in Pep])

    fig, ax = plt.subplots(figsize=(8, 5), dpi=dpi)

    counts, bins, patches = ax.hist(ppm_errors, bins=30, color='#4C72B0', 
                                     alpha=0.7, edgecolor='black', linewidth=0.5)

    mean_ppm = np.mean(ppm_errors)
    std_ppm = np.std(ppm_errors)
    ax.axvline(mean_ppm, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_ppm:.2f} ppm')
    ax.axvline(mean_ppm - std_ppm, color='orange', linestyle=':', linewidth=1.5, label=f'±σ: {std_ppm:.2f} ppm')
    ax.axvline(mean_ppm + std_ppm, color='orange', linestyle=':', linewidth=1.5)

    ax.set_xlabel('PPM Error', fontsize=14, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=14, fontweight='bold')
    ax.set_title('Mass Error Distribution', fontsize=16, fontweight='bold')
    ax.legend(fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Mass error distribution saved")


def ALC_Score_Distribution(DataList, save_dir, FigureName, dpi=300):
    """Distribution of ALC scores"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    alc_scores = np.array([Pep['ALC (%)'] for Pep in DataList if 'ALC (%)' in Pep])

    fig, ax = plt.subplots(figsize=(8, 5), dpi=dpi)

    counts, bins, patches = ax.hist(alc_scores, bins=25, color='#1A9850', 
                                     alpha=0.7, edgecolor='black', linewidth=0.5)

    mean_alc = np.mean(alc_scores)
    ax.axvline(mean_alc, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_alc:.1f}%')

    ax.set_xlabel('ALC Score (%)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=14, fontweight='bold')
    ax.set_title('ALC Score Distribution', fontsize=16, fontweight='bold')
    ax.legend(fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ ALC distribution saved")


# ================================================================
# SCATTERPLOT
# ================================================================

def Scatterplot(DataList, save_dir, FigureName, raw=False, dpi=300):
    """Scatterplot of ppm error vs ALC score"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)
    
    Xvalues = [Pep['ALC (%)'] for Pep in DataList]
    Yvalues = [Pep['ppm'] for Pep in DataList]
    
    fig = plt.figure(figsize=(10, 8), dpi=dpi)
    ax = fig.add_subplot(111)
    
    if raw:
        ax.scatter(Xvalues, Yvalues, marker='D', facecolors='none', edgecolors='green', s=350, alpha=0.35)
    else:
        ax.scatter(Xvalues, Yvalues, marker='D', facecolors='none', edgecolors='blue', s=500, alpha=0.5)
        
    ax.set_xlim(0, 100)                                                        
    ax.set_ylim(-15, 15)                                                        
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))                            
    ax.xaxis.set_minor_locator(AutoMinorLocator(4))                            
    ax.set_ylabel('ppm error', fontsize=32)                                              
    ax.set_xlabel('ALC score (%)', fontsize=32)                                          
    ax.set_title('ppm vs ALC scatterplot', fontsize=36, y=1.04)
    ax.grid(False)                                                      
    ax.set_axisbelow(True)                                                   
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.savefig(file_path, bbox_inches='tight', dpi=dpi)                           
    plt.close()
    print(f"✓ Scatterplot saved")

# ================================================================
# SEQUENCE LOGO PLOTS
# ================================================================

def HistogrammLogo_Overall(data, save_dir, figure_name, dpi=dpi_setting):
    """Overall WebLogo for all sequences with custom colors"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name + ".png")

    cleaned_peptides = list(clean_peptides(data))

    counts_mat = logomaker.alignment_to_matrix(cleaned_peptides)
    prob_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0)
    info_mat = logomaker.transform_matrix(prob_mat, from_type='probability', to_type='information')

    fig, ax = plt.subplots(figsize=(6, 3), dpi=dpi)
    logo = logomaker.Logo(info_mat, ax=ax, font_name='DejaVu Sans', color_scheme=aa_colors)

    for text_obj in ax.texts:
        letter = text_obj.get_text()
        if letter in aa_colors:
            text_obj.set_color(aa_colors[letter])

    ax.set_ylabel("Information (bits)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Position", fontsize=12, fontweight='bold')
    ax.set_title("Sequence Logo (All Sequences)", fontsize=14, fontweight='bold')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Overall sequence logo saved (with custom colors)")


def HistogrammLogo_VariablePositions(data, save_dir, figure_name, variable_pos=variable_positions, dpi=dpi_setting):
    """WebLogo focused on variable positions with custom colors"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name + ".png")

    cleaned_peptides = list(clean_peptides(data))
    
    variable_pos_0based = [pos - 1 for pos in variable_pos]
    variable_sequences = []
    
    for peptide in cleaned_peptides:
        variable_seq = ''.join([peptide[pos] for pos in variable_pos_0based if pos < len(peptide)])
        if len(variable_seq) == len(variable_pos_0based):
            variable_sequences.append(variable_seq)

    if not variable_sequences or len(variable_sequences) == 0:
        print("⚠ Warning: No valid variable positions found")
        return

    counts_mat = logomaker.alignment_to_matrix(variable_sequences)
    prob_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0)
    info_mat = logomaker.transform_matrix(prob_mat, from_type='probability', to_type='information')

    fig, ax = plt.subplots(figsize=(5, 3), dpi=dpi)
    logo = logomaker.Logo(info_mat, ax=ax, font_name='DejaVu Sans', color_scheme=aa_colors)

    for text_obj in ax.texts:
        letter = text_obj.get_text()
        if letter in aa_colors:
            text_obj.set_color(aa_colors[letter])
            text_obj.set_weight('bold')

    ax.set_ylabel("Information (bits)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Variable Position", fontsize=12, fontweight='bold')
    ax.set_title("Sequence Logo (Variable Positions Only)", fontsize=14, fontweight='bold')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Variable positions sequence logo saved (with custom colors)")


def HistogrammLogo_PositionDependent(data, save_dir, figure_name_prefix, dpi=dpi_setting):
    """Create individual WebLogos for each position with custom colors"""
    ensure_directory(save_dir)
    
    cleaned_peptides = list(clean_peptides(data))
    
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ Warning: No peptides found")
        return
        
    max_length = max(len(p) for p in cleaned_peptides)

    fig, axes = plt.subplots(1, max_length, figsize=(max_length * 1.8, 3.5), dpi=dpi)
    
    if max_length == 1:
        axes = [axes]

    for pos in range(max_length):
        position_aas = [peptide[pos] for peptide in cleaned_peptides if pos < len(peptide)]
        
        if not position_aas or len(position_aas) == 0:
            continue
            
        counts_mat = logomaker.alignment_to_matrix(position_aas)
        prob_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0)
        info_mat = logomaker.transform_matrix(prob_mat, from_type='probability', to_type='information')

        logo = logomaker.Logo(info_mat, ax=axes[pos], font_name='DejaVu Sans', color_scheme=aa_colors)

        for text_obj in axes[pos].texts:
            letter = text_obj.get_text()
            if letter in aa_colors:
                text_obj.set_color(aa_colors[letter])

        axes[pos].set_title(f"Pos {pos+1}", fontsize=11, fontweight='bold')
        axes[pos].set_xticks([])
        
        if pos == 0:
            axes[pos].set_ylabel("Bits", fontsize=11, fontweight='bold')
        else:
            axes[pos].set_yticks([])

    plt.tight_layout()
    file_path = os.path.join(save_dir, figure_name_prefix + "_by_position.png")
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Position-dependent sequence logos saved (with custom colors)")
          
# ================================================================
# WEBLOGO FUNCTIONS - PROBABILITY
# ================================================================

def HistogrammLogo_Overall_Probability(data, save_dir, figure_name, dpi=dpi_setting):
    """Overall WebLogo - Probability format (DEFAULT)"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name + ".png")

    cleaned_peptides = list(clean_peptides(data))

    counts_mat = logomaker.alignment_to_matrix(cleaned_peptides)
    prob_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(8, 4), dpi=dpi)
    logo = logomaker.Logo(prob_mat, ax=ax, font_name='DejaVu Sans', color_scheme=aa_colors)

    ax.set_ylabel("Probability", fontsize=12, fontweight='bold')
    ax.set_xlabel("Position", fontsize=12, fontweight='bold')
    ax.set_title("Sequence Logo (All Sequences) - Probability", fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.0)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Overall sequence logo saved (Probability)")


def HistogrammLogo_VariablePositions_Probability(data, save_dir, figure_name, variable_pos=variable_positions, dpi=dpi_setting):
    """WebLogo variable positions - Probability format"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name + ".png")

    cleaned_peptides = list(clean_peptides(data))
    
    variable_pos_0based = [pos - 1 for pos in variable_pos]
    variable_sequences = []
    
    for peptide in cleaned_peptides:
        variable_seq = ''.join([peptide[pos] for pos in variable_pos_0based if pos < len(peptide)])
        if len(variable_seq) == len(variable_pos_0based):
            variable_sequences.append(variable_seq)

    if not variable_sequences or len(variable_sequences) == 0:
        print("⚠ Warning: No valid variable positions found")
        return

    counts_mat = logomaker.alignment_to_matrix(variable_sequences)
    prob_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(6, 4), dpi=dpi)
    logo = logomaker.Logo(prob_mat, ax=ax, font_name='DejaVu Sans', color_scheme=aa_colors)

    ax.set_ylabel("Probability", fontsize=12, fontweight='bold')
    ax.set_xlabel("Position", fontsize=12, fontweight='bold')
    ax.set_title("Sequence Logo (Variable Positions) - Probability", fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.0)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Variable positions sequence logo saved (Probability)")


def HistogrammLogo_PositionDependent_Probability(data, save_dir, figure_name_prefix, dpi=dpi_setting):
    """Individual WebLogos per position - Probability format"""
    ensure_directory(save_dir)
    
    cleaned_peptides = list(clean_peptides(data))
    
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ Warning: No peptides found")
        return
        
    max_length = max(len(p) for p in cleaned_peptides)

    fig, axes = plt.subplots(1, max_length, figsize=(max_length * 2.0, 4), dpi=dpi)
    
    if max_length == 1:
        axes = [axes]

    for pos in range(max_length):
        position_aas = [peptide[pos] for peptide in cleaned_peptides if pos < len(peptide)]
        
        if not position_aas or len(position_aas) == 0:
            continue
            
        counts_mat = logomaker.alignment_to_matrix(position_aas)
        prob_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0)

        logo = logomaker.Logo(prob_mat, ax=axes[pos], font_name='DejaVu Sans', color_scheme=aa_colors)

        axes[pos].set_title(f"Pos {pos+1}", fontsize=12, fontweight='bold')
        axes[pos].set_ylim(0, 1.0)
        axes[pos].set_xticks([])
        
        if pos == 0:
            axes[pos].set_ylabel("Probability", fontsize=11, fontweight='bold')
        else:
            axes[pos].set_yticks([])
            axes[pos].set_ylabel("")

    plt.suptitle("Position-Dependent Sequence Logos (Probability)", fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    file_path = os.path.join(save_dir, figure_name_prefix + "_by_position_probability.png")
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Position-dependent sequence logos saved (Probability)")

# ================================================================
# ALTERNATIVE VISUALIZATION 1: STACKED BAR CHART
# ================================================================

def plot_sequence_stacked_bars(data, save_dir, figure_name, variable_pos=variable_positions, 
                               tracked_aa=None, dpi=dpi_setting):
    """Stacked bar chart showing % of each AA at each position"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    if tracked_aa is None:
        tracked_aa = amino_acids_to_track

    cleaned_peptides = list(clean_peptides(data))
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ No peptides found")
        return

    variable_pos_0based = [pos - 1 for pos in variable_pos]
    aa_colors_dict = create_custom_color_scheme()
    
    position_data = {}
    for pos in variable_pos_0based:
        position_data[pos] = {aa: 0 for aa in tracked_aa}
        
        for peptide in cleaned_peptides:
            if pos < len(peptide):
                aa = peptide[pos]
                if aa in position_data[pos]:
                    position_data[pos][aa] += 1
    
    position_percentages = {}
    for pos, counts in position_data.items():
        total = sum(counts.values())
        position_percentages[pos] = {aa: (count/total)*100 if total > 0 else 0 
                                      for aa, count in counts.items()}

    fig, ax = plt.subplots(figsize=(12, 7), dpi=dpi)

    positions = [f"Pos {p+1}" for p in variable_pos_0based]
    x_pos = np.arange(len(positions))
    
    bottom = np.zeros(len(positions))
    
    for aa in tracked_aa:
        percentages = [position_percentages[pos].get(aa, 0) for pos in variable_pos_0based]
        
        if sum(percentages) > 0:
            ax.bar(x_pos, percentages, bottom=bottom, label=aa, 
                   color=aa_colors_dict.get(aa, '#CCCCCC'), alpha=0.85, edgecolor='black', linewidth=0.8)
            
            for i, (pct, b) in enumerate(zip(percentages, bottom)):
                if pct > 3:
                    ax.text(i, b + pct/2, f'{pct:.0f}%', ha='center', va='center', 
                           fontsize=10, fontweight='bold', color='white')
            
            bottom += np.array(percentages)

    ax.set_ylabel('Percentage (%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Position', fontsize=14, fontweight='bold')
    ax.set_title('Amino Acid Distribution by Position (Stacked %)', fontsize=16, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(positions, fontsize=12)
    ax.set_ylim(0, 105)
    
    ax.legend(title='Amino Acids', bbox_to_anchor=(1.05, 1), loc='upper left', 
              ncol=1, fontsize=11, title_fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Stacked bar chart saved")

# ================================================================
# ALTERNATIVE VISUALIZATION 2: GROUPED BAR CHART
# ================================================================

def plot_sequence_grouped_bars(data, save_dir, figure_name, variable_pos=variable_positions, 
                               tracked_aa=None, dpi=dpi_setting):
    """Grouped bar chart for each position"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    if tracked_aa is None:
        tracked_aa = amino_acids_to_track

    cleaned_peptides = list(clean_peptides(data))
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ No peptides found")
        return

    variable_pos_0based = [pos - 1 for pos in variable_pos]
    aa_colors_dict = create_custom_color_scheme()
    
    position_data = {}
    for pos in variable_pos_0based:
        position_data[pos] = {aa: 0 for aa in tracked_aa}
        
        for peptide in cleaned_peptides:
            if pos < len(peptide):
                aa = peptide[pos]
                if aa in tracked_aa:
                    position_data[pos][aa] += 1
    
    position_percentages = {}
    for pos, counts in position_data.items():
        total = sum(counts.values())
        position_percentages[pos] = {aa: (count/total)*100 if total > 0 else 0 
                                      for aa, count in counts.items()}

    fig, ax = plt.subplots(figsize=(12, 7), dpi=dpi)

    positions = np.arange(len(variable_pos_0based))
    bar_width = 0.2
    
    for idx, aa in enumerate(tracked_aa):
        percentages = [position_percentages[pos].get(aa, 0) for pos in variable_pos_0based]
        offset = (idx - len(tracked_aa)/2 + 0.5) * bar_width
        
        bars = ax.bar(positions + offset, percentages, bar_width, label=aa,
                     color=aa_colors_dict.get(aa, '#CCCCCC'), alpha=0.85, edgecolor='black', linewidth=0.8)
        
        for bar, pct in zip(bars, percentages):
            if pct > 0:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{pct:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_ylabel('Percentage (%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Position', fontsize=14, fontweight='bold')
    ax.set_title('Amino Acid Composition by Position', fontsize=16, fontweight='bold')
    ax.set_xticks(positions)
    ax.set_xticklabels([f'Pos {p+1}' for p in variable_pos_0based], fontsize=12)
    
    max_pct = max([max(pos_pct.values()) for pos_pct in position_percentages.values()])
    ax.set_ylim(0, max_pct * 1.2)
    
    ax.legend(title='Amino Acids', fontsize=11, title_fontsize=12, ncol=len(tracked_aa)//2 + 1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Grouped bar chart saved")

# ================================================================
# ALTERNATIVE VISUALIZATION 3: HEATMAP
# ================================================================

def plot_sequence_heatmap_pwm(data, save_dir, figure_name, variable_pos=variable_positions, 
                              tracked_aa=None, dpi=dpi_setting):
    """Heatmap showing amino acid frequencies"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    if tracked_aa is None:
        tracked_aa = sorted(amino_acids_to_track)

    cleaned_peptides = list(clean_peptides(data))
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ No peptides found")
        return

    variable_pos_0based = [pos - 1 for pos in variable_pos]
    
    freq_matrix = np.zeros((len(tracked_aa), len(variable_pos_0based)))
    
    for col_idx, pos in enumerate(variable_pos_0based):
        position_counts = {aa: 0 for aa in tracked_aa}
        
        for peptide in cleaned_peptides:
            if pos < len(peptide):
                aa = peptide[pos]
                if aa in position_counts:
                    position_counts[aa] += 1
        
        total = sum(position_counts.values())
        for row_idx, aa in enumerate(tracked_aa):
            freq_matrix[row_idx, col_idx] = (position_counts[aa] / total * 100) if total > 0 else 0

    fig, ax = plt.subplots(figsize=(8, 6), dpi=dpi)

    im = ax.imshow(freq_matrix, cmap='YlOrRd', aspect='auto', interpolation='nearest')

    ax.set_xticks(range(len(variable_pos_0based)))
    ax.set_xticklabels([f'Pos {p+1}' for p in variable_pos_0based], fontsize=12, fontweight='bold')
    ax.set_yticks(range(len(tracked_aa)))
    ax.set_yticklabels(tracked_aa, fontsize=12, fontweight='bold')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Frequency (%)', fontsize=12, fontweight='bold')

    for i in range(len(tracked_aa)):
        for j in range(len(variable_pos_0based)):
            value = freq_matrix[i, j]
            if value > 0:
                text_color = 'white' if value > 50 else 'black'
                ax.text(j, i, f'{value:.0f}', ha='center', va='center',
                       color=text_color, fontsize=11, fontweight='bold')

    ax.set_xlabel('Position', fontsize=14, fontweight='bold')
    ax.set_ylabel('Amino Acid', fontsize=14, fontweight='bold')
    ax.set_title('Position Weight Matrix (Heatmap)', fontsize=16, fontweight='bold')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Heatmap saved")

# ================================================================
# ALTERNATIVE VISUALIZATION 4: SEQUENCE ALIGNMENT VIEW
# ================================================================

def plot_sequence_alignment_view(data, save_dir, figure_name, num_sequences=50, dpi=dpi_setting):
    """Shows actual peptide sequences aligned vertically"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    cleaned_peptides = list(clean_peptides(data))
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ No peptides found")
        return

    if len(cleaned_peptides) > num_sequences:
        sample_indices = np.random.choice(len(cleaned_peptides), num_sequences, replace=False)
        cleaned_peptides = [cleaned_peptides[i] for i in sample_indices]

    max_length = max(len(p) for p in cleaned_peptides)
    aa_colors_dict = create_custom_color_scheme()

    fig, ax = plt.subplots(figsize=(12, len(cleaned_peptides) * 0.25 + 2), dpi=dpi)

    for seq_idx, peptide in enumerate(cleaned_peptides):
        for aa_idx, aa in enumerate(peptide):
            color = aa_colors_dict.get(aa, '#CCCCCC')
            rect = plt.Rectangle((aa_idx, seq_idx), 1, 1, 
                               facecolor=color, edgecolor='black', linewidth=0.3, alpha=0.85)
            ax.add_patch(rect)
            
            ax.text(aa_idx + 0.5, seq_idx + 0.5, aa, 
                   ha='center', va='center', fontsize=7, fontweight='bold')

    ax.set_xlim(0, max_length)
    ax.set_ylim(0, len(cleaned_peptides))
    ax.set_aspect('equal')
    ax.invert_yaxis()

    ax.set_xticks(np.arange(0.5, max_length, 1))
    ax.set_xticklabels([f'{i+1}' for i in range(max_length)], fontsize=9, fontweight='bold')
    ax.set_yticks([])
    
    ax.set_xlabel('Position', fontsize=13, fontweight='bold')
    ax.set_title(f'Sequence Alignment View ({len(cleaned_peptides)} sequences)', 
                fontsize=15, fontweight='bold')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Sequence alignment view saved")

# ================================================================
# ALTERNATIVE VISUALIZATION 5: BUBBLE CHART
# ================================================================

def plot_sequence_bubble_chart(data, save_dir, figure_name, variable_pos=variable_positions, 
                               tracked_aa=None, dpi=dpi_setting):
    """Bubble chart visualization"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    if tracked_aa is None:
        tracked_aa = amino_acids_to_track

    cleaned_peptides = list(clean_peptides(data))
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ No peptides found")
        return

    variable_pos_0based = [pos - 1 for pos in variable_pos]
    aa_colors_dict = create_custom_color_scheme()
    
    bubble_data = []
    for pos_idx, pos in enumerate(variable_pos_0based):
        aa_counts = {aa: 0 for aa in tracked_aa}
        
        for peptide in cleaned_peptides:
            if pos < len(peptide):
                aa = peptide[pos]
                if aa in tracked_aa:
                    aa_counts[aa] += 1
        
        total = sum(aa_counts.values())
        for aa_idx, aa in enumerate(tracked_aa):
            pct = (aa_counts[aa] / total * 100) if total > 0 else 0
            bubble_data.append({
                'pos': pos_idx + 1,
                'aa': aa,
                'pct': pct,
                'aa_idx': aa_idx
            })

    fig, ax = plt.subplots(figsize=(10, 6), dpi=dpi)

    for bubble in bubble_data:
        if bubble['pct'] > 0:
            ax.scatter(bubble['pos'], bubble['aa_idx'], 
                      s=bubble['pct'] * 40,
                      color=aa_colors_dict.get(bubble['aa'], '#CCCCCC'),
                      alpha=0.7, edgecolors='black', linewidth=2)
            
            ax.text(bubble['pos'], bubble['aa_idx'], 
                   f'{bubble["pct"]:.0f}%',
                   ha='center', va='center', fontsize=10, fontweight='bold', color='white')

    ax.set_xlabel('Position', fontsize=14, fontweight='bold')
    ax.set_ylabel('Amino Acid', fontsize=14, fontweight='bold')
    ax.set_title('Amino Acid Frequency by Position (Bubble Chart)\nBubble size = % frequency', 
                fontsize=16, fontweight='bold')
    
    ax.set_xticks(range(1, len(variable_pos_0based) + 1))
    ax.set_xticklabels([f'Pos {p+1}' for p in variable_pos_0based], fontsize=12)
    ax.set_yticks(range(len(tracked_aa)))
    ax.set_yticklabels(tracked_aa, fontsize=12, fontweight='bold')
    
    ax.set_xlim(0.5, len(variable_pos_0based) + 0.5)
    ax.set_ylim(-0.5, len(tracked_aa) - 0.5)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Bubble chart saved")

# ================================================================
# ALTERNATIVE VISUALIZATION 6: RADAR CHART
# ================================================================

def plot_sequence_radar_chart(data, save_dir, figure_name, variable_pos=variable_positions, 
                              tracked_aa=None, dpi=dpi_setting):
    """Radar chart for each position"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    if tracked_aa is None:
        tracked_aa = amino_acids_to_track

    cleaned_peptides = list(clean_peptides(data))
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ No peptides found")
        return

    variable_pos_0based = [pos - 1 for pos in variable_pos]
    
    num_pos = len(variable_pos_0based)
    ncols = 2
    nrows = (num_pos + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 5*nrows), 
                             subplot_kw=dict(projection='polar'), dpi=dpi)
    
    if num_pos == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    angles = np.linspace(0, 2*np.pi, len(tracked_aa), endpoint=False).tolist()
    angles += angles[:1]

    for pos_idx, pos in enumerate(variable_pos_0based):
        ax = axes[pos_idx]
        
        aa_counts = {aa: 0 for aa in tracked_aa}
        for peptide in cleaned_peptides:
            if pos < len(peptide):
                aa = peptide[pos]
                if aa in tracked_aa:
                    aa_counts[aa] += 1
        
        total = sum(aa_counts.values())
        values = [(aa_counts[aa] / total * 100) if total > 0 else 0 for aa in tracked_aa]
        values += values[:1]
        
        ax.plot(angles, values, 'o-', linewidth=2.5, color='#4C72B0', markersize=8)
        ax.fill(angles, values, alpha=0.25, color='#4C72B0')
        
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(tracked_aa, fontsize=12, fontweight='bold')
        ax.set_ylim(0, 100)
        ax.set_yticks([20, 40, 60, 80, 100])
        ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], fontsize=9)
        ax.grid(True, linewidth=0.8)
        
        ax.set_title(f'Position {pos+1}', fontsize=14, fontweight='bold', pad=20)
        
        for angle, value, aa in zip(angles[:-1], values[:-1], tracked_aa):
            ax.text(angle, value + 8, f'{value:.0f}%', ha='center', va='center',
                   fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', 
                   facecolor='white', alpha=0.7, edgecolor='none'))

    for idx in range(len(variable_pos_0based), len(axes)):
        axes[idx].remove()

    fig.suptitle('Amino Acid Distribution (Radar View)', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Radar chart saved")

# ================================================================
# ALTERNATIVE VISUALIZATION 8: INFORMATION CONTENT
# ================================================================

def plot_information_content_by_position(data, save_dir, figure_name, variable_pos=variable_positions, 
                                        tracked_aa=None, dpi=dpi_setting):
    """Plot information content at each position"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)
    
    if tracked_aa is None:
        tracked_aa = amino_acids_to_track
    
    cleaned_peptides = list(clean_peptides(data))
    
    if not cleaned_peptides or len(cleaned_peptides) == 0:
        print("⚠ No peptides found")
        return
        
    variable_pos_0based = [pos - 1 for pos in variable_pos]
    
    info_contents = []
    position_labels = []
    
    for pos in variable_pos_0based:
        position_aas = [peptide[pos] for peptide in cleaned_peptides if pos < len(peptide)]
        
        if not position_aas or len(position_aas) == 0:
            info_contents.append(0)
            position_labels.append(f'Pos {pos+1}')
            continue
        
        aa_counts = {}
        for aa in position_aas:
            aa_counts[aa] = aa_counts.get(aa, 0) + 1
        
        total = len(position_aas)
        entropy = 0
        for count in aa_counts.values():
            if count > 0:
                p = count / total
                entropy -= p * np.log2(p)
        
        max_entropy = np.log2(len(tracked_aa))
        info_content = max_entropy - entropy
        info_contents.append(info_content)
        position_labels.append(f'Pos {pos+1}')
    
    fig, ax = plt.subplots(figsize=(11, 6), dpi=dpi)
    
    positions = np.arange(len(variable_pos_0based))
    colors_bar = ['#FF7F0E' if ic > max(info_contents)*0.6 else '#4C72B0' if ic > max(info_contents)*0.3 else '#9467BD' 
                  for ic in info_contents]
    
    bars = ax.bar(positions, info_contents, color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.2)
    
    for bar, ic in zip(bars, info_contents):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{ic:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    ax.set_xlabel('Position', fontsize=14, fontweight='bold')
    ax.set_ylabel('Information Content (bits)', fontsize=14, fontweight='bold')
    ax.set_title('Information Content by Position', fontsize=16, fontweight='bold')
    ax.set_xticks(positions)
    ax.set_xticklabels(position_labels, fontsize=12)
    ax.set_ylim(0, np.log2(len(tracked_aa)) * 1.15)
    
    from matplotlib.patches import Patch
    max_ic = max(info_contents) if info_contents else 1
    legend_elements = [
        Patch(facecolor='#FF7F0E', alpha=0.8, edgecolor='black', label=f'Highly Conserved (>{max_ic*0.6:.2f} bits)'),
        Patch(facecolor='#4C72B0', alpha=0.8, edgecolor='black', label=f'Moderately Conserved ({max_ic*0.3:.2f}-{max_ic*0.6:.2f} bits)'),
        Patch(facecolor='#9467BD', alpha=0.8, edgecolor='black', label=f'Variable (<{max_ic*0.3:.2f} bits)')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=11, title='Conservation Level', title_fontsize=12)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Information content plot saved")

# ================================================================
# POSITION-SPECIFIC FREQUENCY PLOTS
# ================================================================

def calculate_histogram_position_frequencies(peptides, amino_acids=None):
    """Calculate amino acid frequency at each position"""
    if amino_acids is None:
        amino_acids = amino_acids_to_track

    cleaned_peptides = clean_peptides(peptides)
    max_length = max(len(peptide) for peptide in cleaned_peptides)

    position_freqs = {pos: {aa: 0 for aa in amino_acids} for pos in range(max_length)}

    for peptide in cleaned_peptides:
        for pos in range(len(peptide)):
            aa = peptide[pos]
            if aa in amino_acids:
                position_freqs[pos][aa] += 1

    return position_freqs


def plot_position_histogram(position_freqs, save_dir, figure_name, dpi=dpi_setting):
    """Plot histogram of amino acid frequencies at each position"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name + ".png")

    num_positions = len(position_freqs)

    fig, axes = plt.subplots(
        1, num_positions,
        figsize=(num_positions * 1.8, 2.8),
        dpi=dpi,
        sharey=True
    )

    if num_positions == 1:
        axes = [axes]

    for pos, ax in enumerate(axes):
        freqs = position_freqs[pos]

        bars = ax.bar(
            freqs.keys(),
            freqs.values(),
            color='#4C72B0',
            alpha=0.8,
            edgecolor='black',
            linewidth=0.5
        )

        ax.set_title(f"Pos {pos+1}", fontsize=11, fontweight='bold', pad=5)
        ax.set_xticks(range(len(freqs)))
        ax.set_xticklabels(freqs.keys(), rotation=0, fontsize=9)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[0].set_ylabel("Count", fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Position frequency histogram saved")


def plot_position_heatmap(data, save_dir, figure_name, dpi=dpi_setting):
    """Heatmap showing amino acid frequency at each position"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    amino_acids = sorted(amino_acids_to_track)

    cleaned_peptides = clean_peptides(data)
    max_length = max(len(p) for p in cleaned_peptides)

    freq_matrix = np.zeros((len(amino_acids), max_length))

    for peptide in cleaned_peptides:
        for pos in range(len(peptide)):
            aa = peptide[pos]
            if aa in amino_acids:
                aa_idx = amino_acids.index(aa)
                freq_matrix[aa_idx, pos] += 1

    freq_matrix_norm = (freq_matrix / freq_matrix.sum(axis=0)) * 100

    fig, ax = plt.subplots(figsize=(max_length * 0.8 + 2, len(amino_acids) * 0.6 + 1.5), dpi=dpi)

    im = ax.imshow(freq_matrix_norm, cmap='YlOrRd', aspect='auto', interpolation='nearest')

    ax.set_xticks(range(max_length))
    ax.set_xticklabels([f"Pos {i+1}" for i in range(max_length)], fontsize=10)
    ax.set_yticks(range(len(amino_acids)))
    ax.set_yticklabels(amino_acids, fontsize=11)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Frequency (%)', fontsize=11, fontweight='bold')

    for i in range(len(amino_acids)):
        for j in range(max_length):
            text = ax.text(j, i, f'{freq_matrix_norm[i, j]:.0f}',
                          ha="center", va="center", color="black", fontsize=9)

    ax.set_title('Amino Acid Frequency by Position', fontsize=14, fontweight='bold')
    ax.set_xlabel('Position', fontsize=12, fontweight='bold')
    ax.set_ylabel('Amino Acid', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Position heatmap saved")

# ================================================================
# OVERALL AMINO ACID FREQUENCY
# ================================================================

def Histogramm(data, save_dir, figure_name, dpi=dpi_setting, positions=None):
    """Plot overall frequency of amino acids at specified positions"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, figure_name)

    amino_acids = amino_acids_at_VARIABLE_positions
    
    if positions is None:
        positions = variable_positions

    positions = [pos - 1 for pos in positions]

    cleaned_peptides = clean_peptides(data)

    freq = {aa: 0 for aa in amino_acids}

    for peptide in cleaned_peptides:
        for pos in positions:
            if pos < len(peptide):
                aa = peptide[pos]
                if aa in amino_acids:
                    freq[aa] += 1

    total_count = sum(freq.values())
    
    if total_count > 0:
        freq_percent = {aa: (count / total_count) * 100 for aa, count in freq.items()}
    else:
        freq_percent = {aa: 0 for aa in amino_acids}

    fig, ax = plt.subplots(figsize=(10, 7), dpi=dpi)

    bars = ax.bar(
        freq_percent.keys(),
        freq_percent.values(),
        color='#4C72B0',
        alpha=0.85,
        edgecolor='black',
        linewidth=0.5
    )

    theoretical_freq = 100 / len(amino_acids)
    ax.axhline(y=theoretical_freq, color='red', linestyle='--', linewidth=2, 
               label=f'Theoretical ({theoretical_freq:.1f}%)')

    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval, f'{yval:.1f}%', 
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_xlabel('Amino Acid', fontsize=14, fontweight='bold')
    ax.set_ylabel('Frequency (%)', fontsize=14, fontweight='bold')
    ax.set_title('Overall Amino Acid Frequency (Variable Positions)', fontsize=16, fontweight='bold')
    ax.tick_params(axis='both', which='major', labelsize=11)
    ax.legend(fontsize=11, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, bbox_inches='tight', dpi=dpi)
    plt.close()
    print(f"✓ Overall amino acid frequency histogram saved")

# ================================================================
# SEQUENCE LENGTH DISTRIBUTION
# ================================================================

def SequenceLength_Distribution(data, save_dir, FigureName, dpi=dpi_setting):
    """Distribution of sequence lengths"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    cleaned_peptides = clean_peptides(data)
    lengths = [len(p) for p in cleaned_peptides]

    fig, ax = plt.subplots(figsize=(8, 5), dpi=dpi)

    counts, bins, patches = ax.hist(lengths, bins=range(min(lengths), max(lengths)+2), 
                                     color='#9467BD', alpha=0.75, edgecolor='black', linewidth=0.5)

    ax.set_xlabel('Sequence Length', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title('Sequence Length Distribution', fontsize=14, fontweight='bold')
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    stats_text = f"N = {len(lengths)}\nMean = {np.mean(lengths):.1f}\nStd = {np.std(lengths):.2f}"
    ax.text(0.98, 0.97, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Sequence length distribution saved")

# ================================================================
# VENN DIAGRAM
# ================================================================

def plot_venn_diagram(tp_count, fp_count, fn_count, save_dir, FigureName, dpi=dpi_setting):
    """Venn diagram showing TP/FP/FN"""
    try:
        from matplotlib_venn import venn2
        
        ensure_directory(save_dir)
        file_path = os.path.join(save_dir, FigureName)

        fig, ax = plt.subplots(figsize=(8, 6), dpi=dpi)

        venn = venn2(subsets=(fp_count, fn_count, tp_count), 
                     set_labels=('Experimental\n(FP + TP)', 'Theoretical\n(FN + TP)'),
                     ax=ax)

        if venn.get_patch_by_id('10'):
            venn.get_patch_by_id('10').set_color('#FF7F0E')
            venn.get_patch_by_id('10').set_alpha(0.7)
        if venn.get_patch_by_id('01'):
            venn.get_patch_by_id('01').set_color('#1f77b4')
            venn.get_patch_by_id('01').set_alpha(0.7)
        if venn.get_patch_by_id('11'):
            venn.get_patch_by_id('11').set_color('#2ca02c')
            venn.get_patch_by_id('11').set_alpha(0.7)

        ax.set_title('TP/FP/FN Distribution', fontsize=14, fontweight='bold')
        
        stats_text = f"TP = {tp_count}\nFP = {fp_count}\nFN = {fn_count}\n" \
                    f"Precision = {tp_count/(tp_count+fp_count):.2%}\n" \
                    f"Recall = {tp_count/(tp_count+fn_count):.2%}"
        ax.text(0.98, 0.02, stats_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='bottom', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        plt.tight_layout()
        plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        print(f"✓ Venn diagram saved")

    except ImportError:
        print("⚠ matplotlib_venn not installed. Skipping Venn diagram.")

# ================================================================
# QC SUMMARY DASHBOARD
# ================================================================

def plot_qc_summary(DataList, theoretical_masses, save_dir, dpi=dpi_setting):
    """QC Dashboard with integrated panels"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, 'QC_summary_dashboard.png')

    if isinstance(DataList, list):
        data_df = pd.DataFrame(DataList)
    else:
        data_df = DataList

    num_amino_acids = len(allowed_letters)
    
    if num_amino_acids > 6:  #Adjust this number to change between extended and narrow layout for position dependent frequncy bar plots.
        print(f"   ℹ  {num_amino_acids} allowed amino acids detected - using extended dashboard layout")
        fig = plt.figure(figsize=(20, 24), dpi=dpi)
        gs = fig.add_gridspec(6, 3, height_ratios=[1, 1, 0, 1.5, 0, 1.5], 
                      hspace=0.3, wspace=0.35, 
                              left=0.08, right=0.95, top=0.95, bottom=0.08)
        panel_d_gridspec = gs[5, :]
        panel_c_gridspec = gs[3, 0]
        panel_e_gridspec = gs[3, 1:3]
        extended_layout = True
        
    else:
        print(f"   ℹ  {num_amino_acids} allowed amino acids - using standard dashboard layout")
        fig = plt.figure(figsize=(20, 14), dpi=dpi)
        gs = fig.add_gridspec(4, 3, height_ratios=[1, 1, 0.5, 1], 
                              hspace=0, wspace=0.35, 
                              left=0.08, right=0.95, top=0.95, bottom=0.08)
        panel_d_gridspec = gs[3, 1]
        panel_c_gridspec = gs[3, 0]
        panel_e_gridspec = gs[3, 2]
        extended_layout = False

    # PANEL A: MASS DISTRIBUTION KDE
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    
    MassVect = np.array([Pep['Mass'] for Pep in DataList])
    theoretical_masses_arr = np.array(theoretical_masses)
    
    if len(MassVect) > 0:
        Q1_exp, Q3_exp = np.percentile(MassVect, [25, 75])
        IQR_exp = Q3_exp - Q1_exp
        exp_filtered = MassVect[(MassVect >= Q1_exp - 1.5*IQR_exp) & (MassVect <= Q3_exp + 1.5*IQR_exp)]
        
        kde_exp = gaussian_kde(exp_filtered)
        kde_theo = gaussian_kde(theoretical_masses_arr)
        
        x_grid = np.linspace(
            min(exp_filtered.min(), theoretical_masses_arr.min()) * 0.98,
            max(exp_filtered.max(), theoretical_masses_arr.max()) * 1.02,
            1000
        )
        
        y_exp = kde_exp(x_grid)
        y_theo = kde_theo(x_grid)
        
        y_min = np.minimum(y_exp, y_theo)
        
        overlap_area = np.trapz(y_min, x_grid)
        total_area_exp = np.trapz(y_exp, x_grid)
        overlap_percentage = (overlap_area / total_area_exp) * 100
        
        exp_label = f'Experimental ({len(MassVect)})'
        theo_label = f'Theoretical ({len(theoretical_masses_arr)})'
        overlap_label = f'Overlap ({overlap_percentage:.1f}%)'
        
        ax1.plot(x_grid, y_exp, color='#8C564B', linewidth=3.5, label=exp_label)
        ax1.plot(x_grid, y_theo, color='#4C72B0', linewidth=3.5, label=theo_label)
        ax1.fill_between(x_grid, y_min, alpha=0.35, color='#D3D3D3', label=overlap_label)
        
        ax1.set_xlabel('Mass (Da)', fontsize=26, fontweight='bold')
        ax1.set_ylabel('Density', fontsize=26, fontweight='bold')
        ax1.tick_params(axis='both', which='major', labelsize=20)
        ax1.legend(fontsize=18, loc='upper right', framealpha=0.95)
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
    
        ax1.set_title('A', fontsize=48, fontweight='bold', loc='left', pad=15)

    # PANEL B: SEQUENCE LOGO
    ax2 = fig.add_subplot(gs[0:2, 2])
    
    cleaned_peptides = list(clean_peptides(data_df))
    if cleaned_peptides and len(cleaned_peptides) > 0:
        counts_mat = logomaker.alignment_to_matrix(cleaned_peptides)
        prob_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0)
        
        logo = logomaker.Logo(prob_mat, ax=ax2, font_name='DejaVu Sans', color_scheme=aa_colors)
        
        ax2.set_ylabel("Probability", fontsize=26, fontweight='bold')
        ax2.set_xlabel("Position", fontsize=26, fontweight='bold')
        ax2.set_ylim(0, 1.0)
        ax2.tick_params(axis='both', which='major', labelsize=20)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        
        ax2.set_title('B', fontsize=48, fontweight='bold', loc='left', pad=15)

    # PANEL C: AMINO ACID FREQUENCY
    ax3 = fig.add_subplot(panel_c_gridspec)
    
    amino_acids = amino_acids_at_VARIABLE_positions
    positions = [pos - 1 for pos in variable_positions]
    
    freq = {aa: 0 for aa in amino_acids}
    for peptide in cleaned_peptides:
        for pos in positions:
            if pos < len(peptide):
                aa = peptide[pos]
                if aa in amino_acids:
                    freq[aa] += 1
    
    total_count = sum(freq.values())
    if total_count > 0:
        freq_percent = {aa: (count / total_count) * 100 for aa, count in freq.items()}
    else:
        freq_percent = {aa: 0 for aa in amino_acids}
    
    colors_bar = [aa_colors.get(aa, '#CCCCCC') for aa in freq_percent.keys()]
    bars = ax3.bar(freq_percent.keys(), freq_percent.values(), 
                   color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.2)
    
    theoretical_freq = 100 / len(amino_acids)
    ax3.axhline(y=theoretical_freq, color='gray', linestyle=':', linewidth=2.5, alpha=0.7)
    
    ax3.set_ylabel('Frequency (%)', fontsize=26, fontweight='bold')
    ax3.set_xlabel('Amino Acid', fontsize=26, fontweight='bold')
    ax3.tick_params(axis='both', which='major', labelsize=18)
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    ax3.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    ax3.set_title('C', fontsize=48, fontweight='bold', loc='left', pad=15)

    # PANEL D: POSITION-DEPENDENT FREQUENCY
    ax4 = fig.add_subplot(panel_d_gridspec)
    
    if cleaned_peptides and len(cleaned_peptides) > 0:
        position_freqs = calculate_histogram_position_frequencies(data_df, amino_acids_to_track)
        positions_list = sorted(position_freqs.keys())
        aa_list = amino_acids_to_track
        
        if extended_layout:
            from matplotlib.gridspec import GridSpecFromSubplotSpec
            plt.sca(ax4)
            fig.delaxes(ax4)
            
            num_positions = len(positions_list)
            gs_d = GridSpecFromSubplotSpec(1, num_positions, 
                                          subplot_spec=panel_d_gridspec,
                                          hspace=0, wspace=0)
            
            axes_d = [fig.add_subplot(gs_d[0, i]) for i in range(num_positions)]
            
            for pos_idx, pos in enumerate(positions_list):
                ax = axes_d[pos_idx]
                freqs = position_freqs[pos]
                
                bars = ax.bar(
                    freqs.keys(),
                    freqs.values(),
                    color=[aa_colors.get(aa, '#CCCCCC') for aa in freqs.keys()],
                    alpha=0.8,
                    edgecolor='black',
                    linewidth=0.5
                )
                
                ax.set_title(f"Pos {pos+1}", fontsize=20, fontweight='bold', pad=8)
                ax.set_xticks(range(len(freqs)))
                ax.set_xticklabels(freqs.keys(), fontsize=20, fontweight='bold')
                
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                
                if pos_idx == 0:
                    ax.set_ylabel("Count", fontsize=26, fontweight='bold')
                else:
                    ax.set_yticks([])
            
            axes_d[0].text(0, 1.15, 'D', transform=axes_d[0].transAxes,
                          fontsize=48, fontweight='bold', va='top', ha='left')
        
        else:
            ax = ax4
            x = np.arange(len(positions_list))
            width = 0.15
            
            for aa_idx, aa in enumerate(aa_list):
                counts = [position_freqs[pos].get(aa, 0) for pos in positions_list]
                offset = (aa_idx - len(aa_list)/2 + 0.5) * width
                ax.bar(x + offset, counts, width, label=aa, 
                       color=aa_colors.get(aa, '#CCCCCC'), alpha=0.8, edgecolor='black', linewidth=0.8)
            
            ax.set_ylabel('Count', fontsize=26, fontweight='bold')
            ax.set_xlabel('Position', fontsize=22, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels([f'Pos {i+1}' for i in positions_list], fontsize=18)
            ax.tick_params(axis='y', which='major', labelsize=18)
            
            num_entries = len(aa_list)
            if num_entries <= 4:
                legend_fontsize = 16
                legend_ncol = 1
            elif num_entries <= 6:
                legend_fontsize = 14
                legend_ncol = 1
            else:
                legend_fontsize = 8
                legend_ncol = 2
            
            ax.legend(fontsize=legend_fontsize, loc='upper left', ncol=legend_ncol, 
                      framealpha=0, edgecolor='black', fancybox=True,
                      bbox_to_anchor=(1.05, 1))
            
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.grid(True, alpha=0.3, axis='y', linestyle='--')
            
            ax.set_title('D', fontsize=48, fontweight='bold', loc='left', pad=15)

    # PANEL E: SUMMARY TABLE
    ax5 = fig.add_subplot(panel_e_gridspec)
    ax5.axis('off')
    
    total_experimental = len(DataList) if isinstance(DataList, list) else len(DataList)
    total_theoretical = len(theoretical_masses_arr)
    
    recall_percentage = (total_experimental / total_theoretical * 100) if total_theoretical > 0 else 0
    
    table_data = [
        ['Library Size', f'{total_theoretical}'],
        ['Sequenced', f'{total_experimental}'],
        ['Recall (%)', f'{recall_percentage:.1f}']
    ]
    
    table = ax5.table(cellText=table_data, cellLoc='center', loc='center',
                     colWidths=[0.65, 0.35],
                     bbox=[0.02, 0.05, 0.96, 0.90])
    
    table.auto_set_font_size(False)
    table.set_fontsize(26)
    table.scale(1, 4.5)
    
    for i in range(len(table_data)):
        for j in range(2):
            cell = table[(i, j)]
            cell.set_facecolor('white')
            cell.set_text_props(weight='bold', fontsize=26)
            cell.set_edgecolor('none')
            cell.set_linewidth(0)
    
    ax5.set_title('E', fontsize=48, fontweight='bold', loc='left', pad=15)

    plt.savefig(file_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close()
    if extended_layout:
        print(f"✓ QC summary dashboard saved (extended layout for {num_amino_acids} amino acids)")
    else:
        print(f"✓ QC summary dashboard saved (standard layout for {num_amino_acids} amino acids)")

# ================================================================
# STATISTICS AND VALIDATION
# ================================================================

def calculate_exact_matching_and_statistics(theoretical_csv, output_file):
    """Compare peptide sequences and calculate statistics"""
    theoretical_data = pd.read_csv(theoretical_csv)
    experimental_data = pd.read_csv(output_file)
    
    theoretical_data['Peptide'] = theoretical_data['Peptide'].apply(clean_sequence)
    experimental_data['Peptide'] = experimental_data['Peptide'].apply(clean_sequence)
    
    experimental_data = experimental_data.drop_duplicates(subset='Peptide')
    
    theoretical_peptides = set(theoretical_data['Peptide'])
    experimental_peptides = experimental_data['Peptide']

    peptides_df = pd.DataFrame(experimental_data['Peptide'].apply(clean_sequence))
    
    true_positives = set()
    false_positives = set()
    
    for pep_exp in experimental_peptides:
        if pep_exp in theoretical_peptides:
            true_positives.add(pep_exp)
        else:
            false_positives.add(pep_exp)
    
    false_negatives = theoretical_peptides - true_positives
    
    precision = len(true_positives) / (len(true_positives) + len(false_positives)) if (len(true_positives) + len(false_positives)) > 0 else 0
    recall = len(true_positives) / (len(true_positives) + len(false_negatives)) if (len(true_positives) + len(false_negatives)) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    total_sequences = len(experimental_peptides)
    accuracy = len(true_positives) / total_sequences if total_sequences > 0 else 0
    
    total_theoretical_entries = len(theoretical_peptides)
    total_experimental_entries = len(experimental_peptides)
    
    return precision, recall, f1_score, accuracy, len(true_positives), len(false_positives), len(false_negatives), true_positives, total_theoretical_entries, total_experimental_entries, peptides_df


def save_true_positives_to_csv(true_positives, output_csv):
    """Save TP sequences to CSV"""
    df = pd.DataFrame(list(true_positives), columns=['True Positive Sequences'])
    df.to_csv(output_csv, index=False)
    print(f"✓ True Positives saved")


def save_experimental_peptides_to_csv(peptides_df, output_csv):
    """Save experimental peptides to CSV"""
    if 'Peptide' in peptides_df.columns:
        if not peptides_df['Peptide'].dropna().empty:
            peptides_df.to_csv(output_csv, index=False)
            print(f"✓ Experimental peptides saved")


def save_statistics_to_csv(stats, output_csv):
    """Save statistics to CSV"""
    df = pd.DataFrame([stats], columns=['True Positives', 'False Positives', 'False Negatives', 
                                        'Accuracy', 'Recall', 'F1-Score', 'Precision',
                                        'Total Theoretical Entries', 'Total Experimental Entries'])
    df.to_csv(output_csv, index=False)
    print(f"✓ Statistics saved")


def plot_precision_recall(precision, recall, f1_score, save_dir, FigureName, dpi=dpi_setting):
    """Plot precision-recall metrics"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=dpi)

    metrics = ['Precision', 'Recall', 'F1-Score']
    values = [precision, recall, f1_score]
    colors = ['#4C72B0', '#1A9850', '#FF7F0E']

    bars = ax.bar(metrics, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1)

    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1%}', ha='center', va='bottom', fontsize=14, fontweight='bold')

    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Precision-Recall Metrics', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ Precision-recall plot saved")


def plot_tp_fp_fn(tp_count, fp_count, fn_count, save_dir, FigureName, dpi=dpi_setting):
    """Plot TP/FP/FN counts"""
    ensure_directory(save_dir)
    file_path = os.path.join(save_dir, FigureName)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=dpi)

    categories = ['True\nPositives', 'False\nPositives', 'False\nNegatives']
    counts = [tp_count, fp_count, fn_count]
    colors = ['#2ca02c', '#FF7F0E', '#d62728']

    bars = ax.bar(categories, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=1)

    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}', ha='center', va='bottom', fontsize=14, fontweight='bold')

    ax.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax.set_title('TP/FP/FN Distribution', fontsize=14, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.tight_layout()
    plt.savefig(file_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f"✓ TP/FP/FN plot saved")

# ================================================================
# MAIN ANALYSIS PIPELINE
# ================================================================

def main_mass_analysis(output_file, theoretical_csv, save_dir):
    """Main mass distribution analysis"""
    print("\n" + "="*70)
    print("MASS DISTRIBUTION ANALYSIS")
    print("="*70)
    
    data = load_data(output_file)
    output_csv = os.path.join(save_dir, 'theoretical_masses.csv')
    theoretical_masses = load_theoretical_masses(theoretical_csv, output_csv)
    
    Scatterplot(data, save_dir, 'scatterplot_ALC_vs_ppm.png', raw=False)
    MassDistribution_KDE_Improved(data, theoretical_masses, save_dir, 'mass_distribution_KDE.png')
    MassDistribution_Violin(data, theoretical_masses, save_dir, 'mass_distribution_violin.png')
    MassDistribution_Cumulative(data, theoretical_masses, save_dir, 'mass_distribution_CDF.png')
    MassError_Distribution(data, save_dir, 'mass_error_distribution_ppm.png')
    ALC_Score_Distribution(data, save_dir, 'ALC_score_distribution.png')
    
    print("="*70)


def main_sequence_analysis(output_file, save_dir):
    """Main sequence analysis with all visualizations"""
    print("\n" + "="*70)
    print("SEQUENCE ANALYSIS & VISUALIZATIONS")
    print("="*70)
    
    data = pd.read_csv(output_file)

    print("\n1️⃣  Shannon WebLogos (Information Content)...")
    try:
        HistogrammLogo_Overall(data, save_dir, 'logo_overall_sequence_information')
        HistogrammLogo_VariablePositions(data, save_dir, 'logo_variable_positions_information')
        HistogrammLogo_PositionDependent(data, save_dir, 'logo_by_position_information')
        print("   ✓ Information Content logos generated")
    except Exception as e:
        print(f"   ⚠ Information Content error: {e}")

    print("\n2️⃣  WebLogos (Probability Format)...")
    try:
        HistogrammLogo_Overall_Probability(data, save_dir, 'logo_overall_sequence_probability')
        HistogrammLogo_VariablePositions_Probability(data, save_dir, 'logo_variable_positions_probability')
        HistogrammLogo_PositionDependent_Probability(data, save_dir, 'logo_by_position_probability')
        print("   ✓ Probability logos generated")
    except Exception as e:
        print(f"   ⚠ Probability error: {e}")

    print("\n3️⃣  Alternative Visualizations...")
    
    try:
        print("   Generating stacked bars...")
        plot_sequence_stacked_bars(data, save_dir, 'ALT_1_stacked_bars.png', 
                                  variable_pos=variable_positions, 
                                  tracked_aa=amino_acids_to_track)
        print("   ✓ Stacked bar chart")
    except Exception as e:
        print(f"   ⚠ Stacked bars error: {e}")

    try:
        print("   Generating grouped bars...")
        plot_sequence_grouped_bars(data, save_dir, 'ALT_2_grouped_bars.png', 
                                  variable_pos=variable_positions, 
                                  tracked_aa=amino_acids_to_track)
        print("   ✓ Grouped bar chart")
    except Exception as e:
        print(f"   ⚠ Grouped bars error: {e}")

    try:
        print("   Generating heatmap...")
        plot_sequence_heatmap_pwm(data, save_dir, 'ALT_3_heatmap_pwm.png', 
                                 variable_pos=variable_positions, 
                                 tracked_aa=amino_acids_to_track)
        print("   ✓ Heatmap")
    except Exception as e:
        print(f"   ⚠ Heatmap error: {e}")

    try:
        print("   Generating sequence alignment...")
        plot_sequence_alignment_view(data, save_dir, 'ALT_4_sequence_alignment.png', 
                                    num_sequences=50)
        print("   ✓ Sequence alignment view")
    except Exception as e:
        print(f"   ⚠ Sequence alignment error: {e}")

    try:
        print("   Generating bubble chart...")
        plot_sequence_bubble_chart(data, save_dir, 'ALT_5_bubble_chart.png', 
                                  variable_pos=variable_positions, 
                                  tracked_aa=amino_acids_to_track)
        print("   ✓ Bubble chart")
    except Exception as e:
        print(f"   ⚠ Bubble chart error: {e}")

    try:
        print("   Generating radar chart...")
        plot_sequence_radar_chart(data, save_dir, 'ALT_6_radar_chart.png', 
                                 variable_pos=variable_positions, 
                                 tracked_aa=amino_acids_to_track)
        print("   ✓ Radar chart")
    except Exception as e:
        print(f"   ⚠ Radar chart error: {e}")

    try:
        print("   Generating information content...")
        plot_information_content_by_position(data, save_dir, 'ALT_7_information_content.png', 
                                            variable_pos=variable_positions, 
                                            tracked_aa=amino_acids_to_track)
        print("   ✓ Information content plot")
    except Exception as e:
        print(f"   ⚠ Information content error: {e}")

    # ================================================================
    # SECTION 4: TRADITIONAL AMINO ACID HISTOGRAMS
    # ================================================================

    print("\n4️⃣  Traditional histograms...")
    try:
        position_freqs = calculate_histogram_position_frequencies(data)
        plot_position_histogram(position_freqs, save_dir, 'amino_acid_freq_by_position')
        plot_position_heatmap(data, save_dir, 'amino_acid_heatmap_by_position.png')
        Histogramm(data, save_dir, 'amino_acid_frequency_overall_histogram.png')
        print("   ✓ Histograms")
    except Exception as e:
        print(f"   ⚠ Histograms error: {e}")

    print("\n5️⃣  Sequence length...")
    try:
        SequenceLength_Distribution(data, save_dir, 'sequence_length_distribution.png')
        print("   ✓ Length distribution")
    except Exception as e:
        print(f"   ⚠ Length distribution error: {e}")

    print("="*70)


def main_validation_analysis(theoretical_csv, output_file, save_dir):
    """Main validation analysis"""
    print("\n" + "="*70)
    print("VALIDATION & PRECISION-RECALL ANALYSIS")
    print("="*70)
    
    precision, recall, f1_score, accuracy, tp_count, fp_count, fn_count, true_positives, \
    total_theoretical, total_experimental, peptides_df = calculate_exact_matching_and_statistics(
        theoretical_csv, output_file)
    
    stats = {
        'True Positives': tp_count,
        'False Positives': fp_count,
        'False Negatives': fn_count,
        'Accuracy': accuracy,
        'Recall': recall,
        'F1-Score': f1_score,
        'Precision': precision,
        'Total Theoretical Entries': total_theoretical,
        'Total Experimental Entries': total_experimental
    }
    
    save_statistics_to_csv(stats, output_stats_csv)
    save_true_positives_to_csv(true_positives, output_tps_csv)
    save_experimental_peptides_to_csv(peptides_df, output_exp_peptides_csv)

    plot_precision_recall(precision, recall, f1_score, save_dir, 'precision_recall_metrics.png')
    plot_tp_fp_fn(tp_count, fp_count, fn_count, save_dir, 'TP_FP_FN_counts.png')
    
    try:
        plot_venn_diagram(tp_count, fp_count, fn_count, save_dir, 'venn_diagram_TP_FP_FN.png')
    except Exception as e:
        print(f"⚠ Venn diagram: {e}")

    print(f"\n{'RESULTS':-^70}")
    print(f"Total Theoretical Entries:     {total_theoretical}")
    print(f"Total Experimental Entries:    {total_experimental}")
    print(f"\nTrue Positives (TP):           {tp_count}")
    print(f"False Positives (FP):          {fp_count}")
    print(f"False Negatives (FN):          {fn_count}")
    print(f"\nAccuracy:                      {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision:                     {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall (Sensitivity):          {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1-Score:                      {f1_score:.4f}")
    print("="*70)


def main_qc_analysis(output_file, theoretical_csv, save_dir):
    """Generate QC dashboard"""
    print("\n" + "="*70)
    print("QUALITY CONTROL DASHBOARD")
    print("="*70)
    
    data = load_data(output_file)
    output_csv = os.path.join(save_dir, 'theoretical_masses.csv')
    theoretical_masses = load_theoretical_masses(theoretical_csv, output_csv)
    
    plot_qc_summary(data, theoretical_masses, save_dir)
    
    print("="*70)

# ================================================================
# EXECUTION
# ================================================================

if __name__ == "__main__":
    
    print("\n" + "█"*70)
    print("█" + " "*68 + "█")
    print("█" + "PEPTIDE SEQUENCING ANALYSIS PIPELINE".center(68) + "█")
    print("█" + " "*68 + "█")
    print("█"*70)
    
    ensure_directory(save_dir)
    
    # 1. MASS DISTRIBUTION ANALYSIS
    main_mass_analysis(output_file, theoretical_csv, save_dir)
    
    # 2. SEQUENCE ANALYSIS
    main_sequence_analysis(output_file, save_dir)
    
    # 3. VALIDATION ANALYSIS
    main_validation_analysis(theoretical_csv, output_file, save_dir)
    
    # 4. QC DASHBOARD
    main_qc_analysis(output_file, theoretical_csv, save_dir)
    
    print("\n" + "█"*70)
    print("█" + " "*68 + "█")
    print("█" + "🎉 COMPLETE ANALYSIS FINISHED! 🎉".center(68) + "█")
    print("█" + " "*68 + "█")
    print("█"*70)
    print(f"\n✓ All results saved to: {save_dir}")
    print("\n📊 GENERATED VISUALIZATIONS:")
    print("""
    ✅ Mass Distribution:
       • scatterplot_ALC_vs_ppm.png
       • mass_distribution_KDE.png
       • mass_distribution_violin.png
       • mass_distribution_CDF.png
       • mass_error_distribution_ppm.png
       • ALC_score_distribution.png

    ✅ Shannon WebLogos:
       • logo_overall_sequence_information.png
       • logo_variable_positions_information.png
       • logo_by_position_information_by_position.png

    ✅ Probability WebLogos:
       • logo_overall_sequence_probability.png
       • logo_variable_positions_probability.png
       • logo_by_position_probability_by_position.png

    ✅ Alternative Visualizations (7 Methods):
       1️⃣  ALT_1_stacked_bars.png
       2️⃣  ALT_2_grouped_bars.png
       3️⃣  ALT_3_heatmap_pwm.png
       4️⃣  ALT_4_sequence_alignment.png
       5️⃣  ALT_5_bubble_chart.png
       6️⃣  ALT_6_radar_chart.png
       7️⃣  ALT_7_information_content.png

    ✅ Traditional Analysis:
       • amino_acid_freq_by_position.png
       • amino_acid_heatmap_by_position.png
       • amino_acid_frequency_overall_histogram.png
       • sequence_length_distribution.png

    ✅ Validation Metrics:
       • precision_recall_metrics.png
       • TP_FP_FN_counts.png
       • venn_diagram_TP_FP_FN.png

    ✅ Quality Control:
       • QC_summary_dashboard.png

    ✅ Data Files:
       • theoretical_masses.csv
       • output_statistics.csv
       • true_positives.csv
       • experimental_peptides.csv
    """)
    print("\n" + "█"*70 + "\n")